# Secondary Metrics & Significance Test

Three remaining items from the proposal's evaluation protocol, computed against the existing 48-run result files from notebooks 04 (Longformer v2) and 06 (LSTM). No full re-run needed for two of the three -- flagged clearly below where a small exception applies.

**Run order:**
1. Per-ecosystem breakdown -- ready now, zero dependencies
2. Bootstrap significance test for A4 -- ready now, zero dependencies
3. Full-union secondary metric -- **requires one small manual step in between** (re-running just the Spree fold in notebooks 04 and 06) -- see the instructions cell before that section.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/thesis'

Mounted at /content/drive


## 1. Per-Ecosystem Breakdown

**Correction from the original proposal:** it specifies "Java/SpringBoot vs PHP," but Track A ended up spanning Go (Grafana), Python (OpenWebUI, Langflow), Ruby (Spree, Zammad), Node.js (Wekan), and Java (IceCMS) -- not just Java. This reports **Track A (mixed-stack) vs Track B (PHP)** instead, the accurate split given what was actually built. Note this alongside the other documented corrections.

In [2]:
"""
Per-ecosystem breakdown: Track A (mixed-stack: Go/Python/Ruby/Node.js/Java)
vs Track B (PHP), computed from the existing 48 fold/run result files for
both Longformer and LSTM. No new training required.

NOTE: the proposal's original "Java/SpringBoot vs PHP" framing doesn't map
onto Track A, which spans Go, Python, Ruby, Node.js, and Java. This script
reports Track A vs Track B instead -- the accurate split given what was
actually built. Document as a minor correction alongside the others.
"""

import json
import glob
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/content/drive/MyDrive/thesis"

TRACK_B_APPS = {
    "phpapp_ample", "phpapp_lms", "phpapp_quiz", "phpapp_notes",
    "phpapp_filemgmt", "phpapp_oews", "phpapp_leaveborrow",
    "phpapp_studentmgmt", "phpapp_leavemgmt2"
}
TRACK_A_APPS = {
    "Grafana", "IceCMS", "Langflow", "OpenWebUI", "Spree", "Wekan", "Zammad"
}


def load_all_results(results_dir):
    files = sorted(glob.glob(f"{results_dir}/fold_*_run*.json"))
    return [json.load(open(f)) for f in files]


def pooled_metrics(results, label):
    all_probs, all_labels = [], []
    for r in results:
        all_probs.extend(r["probs"])
        all_labels.extend(r["labels"])
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    if len(set(all_labels)) < 2:
        print(f"{label}: only one class present, skipping AUC")
        return

    n_pos = int(all_labels.sum())
    n_neg = len(all_labels) - n_pos
    auc_roc = roc_auc_score(all_labels, all_probs)
    auc_pr = average_precision_score(all_labels, all_probs)

    preds = (all_probs >= 0.5).astype(int)
    tp = int(((preds == 1) & (all_labels == 1)).sum())
    fp = int(((preds == 1) & (all_labels == 0)).sum())
    fn = int(((preds == 0) & (all_labels == 1)).sum())
    tn = int(((preds == 0) & (all_labels == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\n=== {label} ===")
    print(f"n = {len(all_labels)} ({n_pos} pos / {n_neg} neg)")
    print(f"ROC-AUC: {auc_roc:.3f}   PR-AUC: {auc_pr:.3f}")
    print(f"At threshold=0.5: precision={precision:.3f}, recall={recall:.3f}, FPR={fpr:.3f}, F1={f1:.3f}")


def run_breakdown(results_dir, model_name):
    results = load_all_results(results_dir)
    track_a = [r for r in results if r["fold"] in TRACK_A_APPS]
    track_b = [r for r in results if r["fold"] in TRACK_B_APPS]
    unmatched = [r for r in results if r["fold"] not in TRACK_A_APPS and r["fold"] not in TRACK_B_APPS]
    if unmatched:
        print(f"WARNING: {len(unmatched)} results with unmapped fold names: {set(r['fold'] for r in unmatched)}")

    print(f"\n{'#'*60}\n{model_name}\n{'#'*60}")
    pooled_metrics(track_a, f"{model_name} -- Track A (mixed-stack: Go/Python/Ruby/Node.js/Java)")
    pooled_metrics(track_b, f"{model_name} -- Track B (PHP)")
    pooled_metrics(results, f"{model_name} -- Unified (both tracks)")


run_breakdown(f"{BASE}/results/longformer_finetune_v2", "Longformer v2 (PRIMARY)")
run_breakdown(f"{BASE}/results/lstm_comparator", "LSTM Comparator")



############################################################
Longformer v2 (PRIMARY)
############################################################

=== Longformer v2 (PRIMARY) -- Track A (mixed-stack: Go/Python/Ruby/Node.js/Java) ===
n = 240 (30 pos / 210 neg)
ROC-AUC: 0.758   PR-AUC: 0.350
At threshold=0.5: precision=0.366, recall=0.500, FPR=0.124, F1=0.423

=== Longformer v2 (PRIMARY) -- Track B (PHP) ===
n = 306 (66 pos / 240 neg)
ROC-AUC: 0.851   PR-AUC: 0.585
At threshold=0.5: precision=0.522, recall=0.712, FPR=0.179, F1=0.603

=== Longformer v2 (PRIMARY) -- Unified (both tracks) ===
n = 546 (96 pos / 450 neg)
ROC-AUC: 0.827   PR-AUC: 0.511
At threshold=0.5: precision=0.473, recall=0.646, FPR=0.153, F1=0.546

############################################################
LSTM Comparator
############################################################

=== LSTM Comparator -- Track A (mixed-stack: Go/Python/Ruby/Node.js/Java) ===
n = 240 (30 pos / 210 neg)
ROC-AUC: 0.920   PR-AUC: 0.578
A

## 2. Bootstrap Significance Test (catalogue angle A4)

Tests whether the LSTM vs Longformer F1 difference meets the catalogue's threshold: **|difference| >= 0.05 with a 95% CI excluding zero**. Uses a paired bootstrap (same resampled indices for both models) since both were evaluated on identical inputs in identical order -- more statistically appropriate than an independent-samples test.

In [3]:
"""
Bootstrap significance test for catalogue angle A4: is the LSTM vs
Longformer F1 difference significant (95% CI excluding zero, |diff| >= 0.05)?

Paired bootstrap: both models were evaluated on identical eval sets per
fold, in the same order, so this resamples matched indices to preserve
the pairing -- more powerful and correct than an independent-samples test.

Run in Colab after both aggregation scripts (notebooks 04 and 06) have
produced pooled_predictions.npz.
"""

import numpy as np

BASE = "/content/drive/MyDrive/thesis"

longformer = np.load(f"{BASE}/results/longformer_finetune_v2/pooled_predictions.npz")
lstm = np.load(f"{BASE}/results/lstm_comparator/pooled_predictions.npz")

lf_probs, lf_labels = longformer["probs"], longformer["labels"]
lstm_probs, lstm_labels = lstm["probs"], lstm["labels"]

assert len(lf_probs) == len(lstm_probs), "Pooled arrays are different lengths -- not directly comparable!"
assert np.array_equal(lf_labels, lstm_labels), "Label arrays don't match -- pooling order may differ between models!"

labels = lf_labels  # identical for both, per the assertion above


def f1_at_threshold(probs, labels, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    tp = ((preds == 1) & (labels == 1)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0


n = len(labels)
n_bootstrap = 10000
rng = np.random.default_rng(42)

lf_f1_point = f1_at_threshold(lf_probs, labels)
lstm_f1_point = f1_at_threshold(lstm_probs, labels)
diff_point = lstm_f1_point - lf_f1_point

diffs = np.empty(n_bootstrap)
for i in range(n_bootstrap):
    idx = rng.integers(0, n, size=n)  # paired resample -- same indices for both models
    lf_f1 = f1_at_threshold(lf_probs[idx], labels[idx])
    lstm_f1 = f1_at_threshold(lstm_probs[idx], labels[idx])
    diffs[i] = lstm_f1 - lf_f1

ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])

print(f"Point estimate F1: Longformer={lf_f1_point:.3f}, LSTM={lstm_f1_point:.3f}")
print(f"Point estimate difference (LSTM - Longformer): {diff_point:.3f}")
print(f"95% Bootstrap CI on difference: [{ci_low:.3f}, {ci_high:.3f}]")
print(f"CI excludes zero: {ci_low > 0 or ci_high < 0}")
print(f"\nCatalogue A4 threshold: |difference| >= 0.05 with 95% CI excluding zero")
print(f"Meets threshold: {abs(diff_point) >= 0.05 and (ci_low > 0 or ci_high < 0)}")


Point estimate F1: Longformer=0.546, LSTM=0.756
Point estimate difference (LSTM - Longformer): 0.210
95% Bootstrap CI on difference: [0.127, 0.294]
CI excludes zero: True

Catalogue A4 threshold: |difference| >= 0.05 with 95% CI excluding zero
Meets threshold: True


## 3. Full-(30/31)-CVE-Union Secondary Metric

**Constraint:** the fine-tuning scripts in notebooks 04/06 deliberately discard the trained model after each fold (`del model, trainer`) to save storage -- only predictions were kept. This means there's no trained model available to retroactively score a *new* case against.

`CVE_2026_25757` (the excluded single-step Spree case) can still be added cheaply: the `Spree` fold already holds the Spree app out entirely, so adding this case to that fold's eval set and re-running **just that one fold** (not all 16) picks it up.

`CVE_2023_43663` (PrestaShop) has **no valid exemplar** at all (all 7 candidate findings were false positives) -- it is NOT added. Report this metric as **30 of 31 confirmed cases**, with `CVE_2023_43663` documented as excluded.

### Step 3a -- add the single-step case to the Spree fold's eval set (run once)

In [4]:
"""
Adds CVE_2026_25757 (single-step Spree case) to the Spree fold's eval set,
so the full-union secondary metric can include it. Run ONCE before
re-running the Spree fold (Longformer + LSTM, 3 runs each).

CVE_2023_43663 is NOT added -- no valid exemplar exists (all 7 candidate
findings were false positives). Report the union metric as 30/31 cases,
with CVE_2023_43663 documented as excluded.
"""

import json

BASE = "/content/drive/MyDrive/thesis"
FOLD_DIR = f"{BASE}/data/loao_folds_v2/fold_Spree"

# Single-step exemplar: no precondition exists for this case (order number
# already known from checkout response), so this is a single-request
# sequence, not a [SEP]-joined pair.
single_step_record = {
    "sequence": "METHOD:GET PATH:/api/v1/orders/{val} ROLE:Anonymous",
    "label": 1
}

eval_path = f"{FOLD_DIR}/eval.jsonl"
with open(eval_path) as f:
    existing = [json.loads(line) for line in f]

print(f"Existing eval records: {len(existing)}")
existing.append(single_step_record)

with open(eval_path, "w") as f:
    for r in existing:
        f.write(json.dumps(r) + "\n")

print(f"Added CVE_2026_25757 (single-step). New eval count: {len(existing)}")
print("Now re-run ONLY the Spree fold (Longformer v2 script + LSTM script,")
print("restricting fold_names = ['Spree'] in each, 3 runs each) to pick up")
print("this addition, then re-run the union aggregation below.")


Existing eval records: 11
Added CVE_2026_25757 (single-step). New eval count: 12
Now re-run ONLY the Spree fold (Longformer v2 script + LSTM script,
restricting fold_names = ['Spree'] in each, 3 runs each) to pick up
this addition, then re-run the union aggregation below.


### Step 3b -- MANUAL STEP REQUIRED before continuing

Go to **notebook `04_longformer_finetuning_v2_PRIMARY`**, in the fine-tuning cell find:
```python
fold_names = sorted(
    d.replace("fold_", "") for d in os.listdir(FOLDS_DIR) if d.startswith("fold_")
)
```
and temporarily replace it with:
```python
fold_names = ["Spree"]
```
Run that cell (3 runs, fast -- only one fold). This overwrites the 3 existing `fold_Spree_run*.json` files with predictions that include the new single-step case.

Do the **same** in notebook `06_lstm_comparator` (same `fold_names` override, same fold).

Once both are done, come back here and run Step 3c below.

### Step 3c -- aggregate the full-union metric (run after Step 3b is complete)

In [5]:
"""
Full-(30 of 31)-CVE-union secondary metric. Run AFTER re-running the Spree
fold with the single-step case added (see add_spree_single_step.py).

Pools all fold results as-is -- since the Spree fold files were overwritten
with the expanded eval set, this automatically includes CVE_2026_25757.
CVE_2023_43663 remains excluded (no valid exemplar).
"""

import json
import glob
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve

BASE = "/content/drive/MyDrive/thesis"


def load_all_results(results_dir):
    files = sorted(glob.glob(f"{results_dir}/fold_*_run*.json"))
    return [json.load(open(f)) for f in files]


def report(results_dir, model_name):
    results = load_all_results(results_dir)
    all_probs, all_labels = [], []
    for r in results:
        all_probs.extend(r["probs"])
        all_labels.extend(r["labels"])
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    n_pos = int(all_labels.sum())
    n_neg = len(all_labels) - n_pos
    auc_roc = roc_auc_score(all_labels, all_probs)
    auc_pr = average_precision_score(all_labels, all_probs)

    preds = (all_probs >= 0.5).astype(int)
    tp = int(((preds == 1) & (all_labels == 1)).sum())
    fp = int(((preds == 1) & (all_labels == 0)).sum())
    fn = int(((preds == 0) & (all_labels == 1)).sum())
    tn = int(((preds == 0) & (all_labels == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\n=== {model_name} -- Full union (30/31 confirmed cases; CVE_2023_43663 excluded, no valid exemplar) ===")
    print(f"n = {len(all_labels)} ({n_pos} pos / {n_neg} neg)")
    print(f"ROC-AUC: {auc_roc:.3f}   PR-AUC: {auc_pr:.3f}")
    print(f"At threshold=0.5: precision={precision:.3f}, recall={recall:.3f}, FPR={fpr:.3f}, F1={f1:.3f}")


report(f"{BASE}/results/longformer_finetune_v2", "Longformer v2 (PRIMARY)")
report(f"{BASE}/results/lstm_comparator", "LSTM Comparator")



=== Longformer v2 (PRIMARY) -- Full union (30/31 confirmed cases; CVE_2023_43663 excluded, no valid exemplar) ===
n = 549 (99 pos / 450 neg)
ROC-AUC: 0.811   PR-AUC: 0.485
At threshold=0.5: precision=0.438, recall=0.646, FPR=0.182, F1=0.522

=== LSTM Comparator -- Full union (30/31 confirmed cases; CVE_2023_43663 excluded, no valid exemplar) ===
n = 549 (99 pos / 450 neg)
ROC-AUC: 0.942   PR-AUC: 0.719
At threshold=0.5: precision=0.696, recall=0.808, FPR=0.078, F1=0.748
